# 01 - Data Understanding
**Project:** Spotify Music Analytics - End-to-End Data Analysis

This notebook loads the raw Spotify tracks CSV and inspects it **without changing it**.
Everything shown is computed from your file - nothing here is pre-written output.

**Before running:** place the Kaggle CSV in `dataset/raw/` (see `docs/setup_guide.md`).

In [ ]:
import sys, warnings
from pathlib import Path

# Make `src` importable whether Jupyter starts in the project root or in notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve_path

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

cfg = load_config()

## A. Load the dataset
Set `SAMPLE_ROWS` to a number (e.g. `100_000`) for a quick trial on a huge file; keep `None` for everything.

In [ ]:
from src.data_loader import find_raw_file, read_raw_csv, validate_schema

SAMPLE_ROWS = None
raw_path = find_raw_file(cfg)
raw = read_raw_csv(raw_path, cfg["column_aliases"], cfg["loading"]["chunk_size"], SAMPLE_ROWS)
print(f"Loaded: {raw_path.name}")

## B. Schema validation
Column names are normalised (lower-case, aliases such as `artist_name` -> `artists`). Required columns must exist; optional ones (`album_name`, `explicit`, `year`) differ between datasets.

In [ ]:
schema = validate_schema(raw)
print("Schema OK:", schema["ok"])
print("Missing required columns:", schema["missing_required"])
print("Missing optional columns:", schema["missing_optional"])

## C. First and last records

In [ ]:
raw.head()

In [ ]:
raw.tail()

## D. Dimensions

In [ ]:
rows, cols = raw.shape
print(f"Rows: {rows:,}\nColumns: {cols}")
print(f"Approx. memory: {raw.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

## E. Column information

In [ ]:
raw.info()

## F. Descriptive statistics

In [ ]:
raw.describe().T

In [ ]:
raw.describe(include=['object', 'string']).T

## G. Missing values

In [ ]:
missing = raw.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(raw) * 100).round(3)
missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)

In [ ]:
if missing["missing_count"].sum() == 0:
    print("No missing values detected in the loaded columns.")

## H. Duplicate records

In [ ]:
exact_duplicates = int(raw.duplicated().sum())
pair_duplicates = int(raw.duplicated(subset=["track_id", "track_genre"]).sum())
ids_in_many_genres = int((raw.groupby("track_id")["track_genre"].nunique() > 1).sum())
print(f"Exact duplicate rows:                 {exact_duplicates:,}")
print(f"Duplicate (track_id, track_genre):    {pair_duplicates:,}")
print(f"track_ids that appear in >1 genre:    {ids_in_many_genres:,}  (informational - kept as separate rows)")

## I. Quick sanity checks on value ranges (before cleaning)

In [ ]:
numeric = raw.apply(pd.to_numeric, errors="coerce")
checks = {
    "popularity outside 0-100": int((~numeric["popularity"].between(0, 100) & numeric["popularity"].notna()).sum()),
    "duration_ms <= 0": int((numeric["duration_ms"] <= 0).sum()),
    "danceability outside 0-1": int((~numeric["danceability"].between(0, 1) & numeric["danceability"].notna()).sum()),
    "energy outside 0-1": int((~numeric["energy"].between(0, 1) & numeric["energy"].notna()).sum()),
    "tempo == 0": int((numeric["tempo"] == 0).sum()),
}
pd.Series(checks, name="rows").to_frame()

**Next:** `02_data_cleaning.ipynb` applies the cleaning rules and saves the cleaned dataset.